# 01 · 1D-ResNet18 Baseline

Part of the study *Multi-Label 12-Lead ECG Classification on the PTB-XL Dataset* ([arXiv:2609.12803](https://arxiv.org/abs/2609.12803)). Run order, folder layout, and results: [project README](../README.md).

> GPU: T4. If you run out of memory, lower `batch_size`.

Shared protocol (`ecg_common.py`): official PTB-XL folds 1–8 / 9 / 10, checkpoint = validation macro-AUPRC, val F1-max thresholds, reported metrics = test fold 10. Outputs: `experiments/<model>/seed_<seed>/` and `results/<canonical>/seed_<seed>/`.
Reads `processed/sr100` from notebook 00.


In [ ]:
!pip install -q openpyxl scikit-learn pandas numpy matplotlib seaborn torch tqdm


In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')

SAMPLING_RATE = 100
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'ECG AI'
PROCESSED_DIR = PROJECT_DIR / 'processed' / f'sr{SAMPLING_RATE}'
EXPERIMENTS_DIR = PROJECT_DIR / 'experiments'
RESULTS_DIR = PROJECT_DIR / 'results'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import load_processed_manifest

_manifest = load_processed_manifest(PROCESSED_DIR)
print('processed:', PROCESSED_DIR)
print('dataset_version:', _manifest.get('dataset_version') or _manifest.get('raw_root_name'))
print('split:', _manifest.get('split_by'), _manifest.get('train_folds'), _manifest.get('val_folds'), _manifest.get('test_folds'))


## Shared protocol

`notebooks/ecg_common.py` is imported. Checkpoint / early stop: **validation macro-AUPRC**. Architecture stays in this notebook; splits, metrics, thresholds, and the Trainer live in the shared module.


In [ ]:
from __future__ import annotations

import sys

import numpy as np
import pandas as pd
import torch
from torch import nn

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    CLASSES,
    EARLY_STOPPING_PATIENCE,
    EVALUATE_TEST as DEFAULT_EVALUATE_TEST,
    ExperimentConfig,
    MAX_EPOCHS,
    PRIMARY_MONITOR,
    SEEDS,
    build_trainer,
    finalize_run_config,
)

EVALUATE_TEST = DEFAULT_EVALUATE_TEST  # True: score test fold 10 with val-tuned thresholds
ACTIVE_SEEDS = [42]  # paper: ACTIVE_SEEDS = SEEDS
print('monitor=', PRIMARY_MONITOR, '| epochs=', MAX_EPOCHS, '| patience=', EARLY_STOPPING_PATIENCE)
print('ACTIVE_SEEDS', ACTIVE_SEEDS, '| EVALUATE_TEST', EVALUATE_TEST)


## Model

Keep architecture hyperparameters (`dropout`, `base_channels`) in the `model_factory` below **and** in `experiment.extra` together; changing only one side does not update the model.


In [ ]:
# MODEL_ARCHITECTURE
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class ResNet1D18(nn.Module):
    def __init__(self, in_channels=12, num_classes=5, base_channels=64, dropout=0.2) -> None:
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_channels, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        channels = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.layers = nn.Sequential(
            self._make_layer(channels[0], channels[0], blocks=2, stride=1),
            self._make_layer(channels[0], channels[1], blocks=2, stride=2),
            self._make_layer(channels[1], channels[2], blocks=2, stride=2),
            self._make_layer(channels[2], channels[3], blocks=2, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(channels[3], num_classes))

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = [ResidualBlock1D(in_channels, out_channels, stride=stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock1D(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x)
        return self.head(x)


## Training hyperparameters

Hyperparameters are defined in `ExperimentConfig(...)` in the next cell.

ResNet18-1D: `batch_size=64`, `learning_rate=1e-3`. Shared protocol: `MAX_EPOCHS=50`, `EARLY_STOPPING_PATIENCE=7`, checkpoint = validation **macro-AUPRC**. `ACTIVE_SEEDS = [42]` is a single seed; for the paper use `ACTIVE_SEEDS = SEEDS` (`[42, 52, 62]`).
Checkpoints and thresholds are fit on val fold 9; reported performance analysis uses test fold 10.


In [ ]:
experiment = ExperimentConfig(
    model_name='resnet18_1d',
    processed_dir=str(PROCESSED_DIR),
    output_root=str(EXPERIMENTS_DIR),
    learning_rate=1e-3,
    weight_decay=1e-4,
    epochs=MAX_EPOCHS,
    batch_size=64,
    sampling_rate=SAMPLING_RATE,
    seed=42,
    patience=EARLY_STOPPING_PATIENCE,
    monitor=PRIMARY_MONITOR,
    evaluate_test=EVALUATE_TEST,
    results_root=str(RESULTS_DIR),
    extra={'dropout': 0.2, 'base_channels': 64},
)

def model_factory(num_classes: int):
    return ResNet1D18(in_channels=12, num_classes=num_classes, base_channels=64, dropout=0.2)

print('config:', experiment.model_name, '| lr', experiment.learning_rate, '| batch', experiment.batch_size, '| epochs', experiment.epochs)
print('X_train', tuple(np.load(Path(experiment.processed_dir) / 'X_train.npy', mmap_mode='r').shape))


In [ ]:
for seed in ACTIVE_SEEDS:
    experiment.seed = int(seed)
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)
    history_df = trainer.fit(
        data_module.train_dataloader(),
        data_module.val_dataloader(),
        epochs=experiment.epochs,
    )
    protocol = trainer.evaluate_with_protocol(
        data_module.val_dataloader(),
        test_loader=data_module.test_dataloader(),
        evaluate_test=EVALUATE_TEST,
        results_root=RESULTS_DIR,
    )
    finalize_run_config(
        trainer,
        extra={'dataset_version': data_module.manifest.get('dataset_version') or data_module.manifest.get('raw_root_name')},
    )
    print('best val macro-AUPRC', protocol.get('best_validation_macro_auprc'), 'epoch', protocol.get('best_epoch'))

history_df.tail()


## Resume from last.pt

If the `fit` cell above was interrupted (Colab disconnect, OOM, runtime reset), run this cell. It loads `last.pt` (weights + optimizer) and writes remaining epochs into the same run folder.

After a kernel restart, skip the training cell: re-run the install, Drive, protocol, model, and `ExperimentConfig` cells, then this cell.
If `trainer` is missing, this cell rebuilds it with `build_trainer`. Do not point it at another model's `last.pt`.


In [ ]:
import json

RESUME_RUN_DIR = None  # None: experiments/resnet18_1d/seed_<seed>

if 'trainer' not in globals() or 'data_module' not in globals():
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)

_resume_dir = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else (
    Path(experiment.output_root) / experiment.model_name / f'seed_{int(experiment.seed)}'
)
_cfg_path = _resume_dir / 'config.json'
if _cfg_path.exists():
    _cfg = json.loads(_cfg_path.read_text(encoding='utf-8'))
    saved = str(_cfg.get('model_name') or '')
    if saved and saved != experiment.model_name:
        raise ValueError(
            f'Refuse resume: {_resume_dir} belongs to {saved}, this notebook is {experiment.model_name}.'
        )
if not (_resume_dir / 'checkpoints' / 'last.pt').exists():
    raise FileNotFoundError(f'last.pt not found: {_resume_dir / "checkpoints" / "last.pt"}')

history_df = trainer.fit(
    data_module.train_dataloader(),
    data_module.val_dataloader(),
    epochs=experiment.epochs,
    resume_from_last=True,
    resume_run_dir=str(_resume_dir),
)
protocol = trainer.evaluate_with_protocol(
    data_module.val_dataloader(),
    test_loader=data_module.test_dataloader(),
    evaluate_test=EVALUATE_TEST,
    results_root=RESULTS_DIR,
)
history_df.tail()


## Performance analysis

This section is **standalone**: after a runtime restart it can be run on its own, without the install, import, or training cells above.

Loads `experiments/<model>/seed_42/checkpoints/best.pt`. Thresholds are fit on val fold 9 (F1-max). Reported metrics, 2×2 confusion matrices, and prediction CSVs use **test fold 10**.
Files: `experiments/.../predictions/val_predictions.csv` and `test_predictions.csv`; copy under `results/<canonical>/seed_42/`.


In [ ]:
# ANALYZE_FROM_BEST_PT
# This cell runs standalone. Cells above (pip / imports / training) are not required.
import sys
import subprocess
from pathlib import Path

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'openpyxl',
        'scikit-learn',
        'pandas',
        'numpy',
        'matplotlib',
        'seaborn',
        'torch',
        'tqdm',
    ]
)
try:
    from google.colab import drive

    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive') / 'ECG AI'
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    evaluate_and_display_best_pt,
    exec_notebook_cells,
    extra_from_run_config,
    load_json,
    seed_run_dir,
)

EVALUATE_TEST = True  # always score test fold 10 from best.pt
ANALYSIS_SEED = 42
MODEL_NAME = 'resnet18_1d'
NOTEBOOK_PATH = PROJECT_DIR / 'notebooks' / '01_resnet18_1d_baseline.ipynb'
run_dir = seed_run_dir(PROJECT_DIR / 'experiments', MODEL_NAME, ANALYSIS_SEED)
ckpt = run_dir / 'checkpoints' / 'best.pt'
if not ckpt.exists():
    raise FileNotFoundError(f'best.pt not found: {ckpt}')

cfg = load_json(run_dir / 'config.json') if (run_dir / 'config.json').exists() else {}
extra = extra_from_run_config(cfg)

import torch
from torch import nn

exec_notebook_cells(NOTEBOOK_PATH, '# MODEL_ARCHITECTURE', globals())

def model_factory(num_classes: int):
    return ResNet1D18(
        in_channels=12,
        num_classes=num_classes,
        base_channels=int(extra.get('base_channels', 64)),
        dropout=float(extra.get('dropout', 0.2)),
    )

evaluate_and_display_best_pt(
    model_factory,
    run_dir,
    processed_dir=PROJECT_DIR / 'processed' / 'sr100',
    results_root=PROJECT_DIR / 'results',
    evaluate_test=True,
)
